# Задание
Вариант 3. Квантование весов для NPU
Реализуйте функцию квантования весов нейронной сети с плавающей точкой (float32) в формат int8 для использования в NPU. Создайте класс QuantizedLayer, который хранит квантованные веса и выполняет прямое распространение с деквантованием. Сравните точность до и после квантования на простых данных.

In [ ]:
import numpy as np

def quantize_weights(weights):

    max_val = np.max(np.abs(weights))
    scale = max_val / 127
    q_weights = np.round(weights / scale).astype(np.int8)
    return q_weights, scale

def dequantize_weights(q_weights, scale):
    return q_weights.astype(np.float32) * scale

class QuantizedLayer:
    def __init__(self, weights):
        self.q_weights, self.scale = quantize_weights(weights)
    
    def forward(self, x):
        w = dequantize_weights(self.q_weights, self.scale)
        return x @ w 

np.random.seed(42)
X = np.random.randn(5, 4).astype(np.float32)  
W = np.random.randn(4, 3).astype(np.float32)  
y_true = np.random.randn(5, 3).astype(np.float32)

y_float = X @ W

q_layer = QuantizedLayer(W)
y_quant = q_layer.forward(X)

mse = np.mean((y_float - y_quant)**2)
print("Исходные веса:\n", W)
print("Квантованные веса:\n", q_layer.q_weights)
print("MSE между float и квантованным forward:", mse)

Исходные веса:
 [[ 1.4656488  -0.2257763   0.0675282 ]
 [-1.4247482  -0.54438275  0.11092259]
 [-1.1509936   0.37569803 -0.6006387 ]
 [-0.29169375 -0.6017066   1.8522782 ]]
Квантованные веса:
 [[100 -15   5]
 [-98 -37   8]
 [-79  26 -41]
 [-20 -41 127]]
MSE между float и квантованным forward: 6.717196e-05
